# Validação dos atributos das entidades

## Introdução

Após confirmar que as entidades previstas nos requisitos estão representadas no
dataset, esta etapa avalia os atributos disponíveis em cada uma delas.

A validação busca verificar se os campos são suficientes para representar o
domínio, se possuem qualidade adequada e se podem apoiar a definição de
identificadores, relacionamentos e regras de integridade no modelo conceitual.

## Objetivos

- comparar os atributos do dataset com o dicionário de dados dos requisitos;
- identificar atributos ausentes, adicionais ou com nomenclatura divergente;
- analisar tipos inferidos, preenchimento e unicidade;
- levantar candidatos a identificadores e chaves;
- registrar decisões de inclusão, transformação ou descarte de atributos.

## 1. Critérios de validação

Cada atributo será avaliado conforme os seguintes critérios:

| Critério | Pergunta de validação |
|---|---|
| Aderência | O atributo está previsto nos requisitos ou é relevante ao domínio? |
| Completude | A quantidade de valores ausentes compromete seu uso? |
| Unicidade | O campo pode identificar registros sozinho ou em conjunto? |
| Tipo | O tipo inferido é compatível com seu significado? |
| Nomenclatura | O nome representa o conceito de forma clara e consistente? |
| Decisão | O atributo será mantido, transformado, derivado ou descartado? |

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

data_directories = (Path("data/raw"), Path("../../data/raw"))
data_dir = next((path for path in data_directories if path.is_dir()), None)

if data_dir is None:
    raise FileNotFoundError(
        "Diretório data/raw não encontrado. Consulte data/README.md para obter os dados."
    )

## 2. Carregamento das entidades

Nesta análise são consideradas as sete entidades de domínio validadas na etapa
anterior. As tabelas auxiliares de geolocalização e tradução de categorias serão
avaliadas separadamente quando seus relacionamentos forem definidos.

In [2]:
arquivos_entidades = {
    "Clientes": "olist_customers_dataset.csv",
    "Pedidos": "olist_orders_dataset.csv",
    "Itens do Pedido": "olist_order_items_dataset.csv",
    "Produtos": "olist_products_dataset.csv",
    "Vendedores": "olist_sellers_dataset.csv",
    "Pagamentos": "olist_order_payments_dataset.csv",
    "Avaliações": "olist_order_reviews_dataset.csv",
}

entidades = {
    nome: pd.read_csv(data_dir / arquivo)
    for nome, arquivo in arquivos_entidades.items()
}

## 3. Inventário e aderência aos requisitos

Os atributos documentados nos requisitos são comparados com os campos existentes
no dataset. Campos adicionais são avaliados pelo valor que agregam ao domínio;
campos textuais livres são descartados por estarem fora do escopo definido.

In [3]:
atributos_esperados = {
    "Clientes": {
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
    },
    "Pedidos": {
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    },
    "Itens do Pedido": {
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value",
    },
    "Produtos": {
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    },
    "Vendedores": {
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state",
    },
    "Pagamentos": {
        "order_id",
        "payment_type",
        "payment_installments",
        "payment_value",
    },
    "Avaliações": {
        "review_id",
        "order_id",
        "review_score",
        "review_creation_date",
    },
}

atributos_fora_do_escopo = {
    ("Avaliações", "review_comment_title"),
    ("Avaliações", "review_comment_message"),
}

justificativas_adicionais = {
    ("Itens do Pedido", "shipping_limit_date"): (
        "Apoia a análise do prazo operacional de expedição."
    ),
    ("Produtos", "product_name_lenght"): (
        "Metadado descritivo opcional para análises de catálogo."
    ),
    ("Produtos", "product_description_lenght"): (
        "Metadado descritivo opcional para análises de catálogo."
    ),
    ("Produtos", "product_photos_qty"): (
        "Metadado opcional sobre a apresentação do produto."
    ),
    ("Pagamentos", "payment_sequential"): (
        "Necessário para distinguir pagamentos do mesmo pedido e compor a chave."
    ),
    ("Avaliações", "review_answer_timestamp"): (
        "Permite analisar o tempo de resposta à avaliação."
    ),
}


def tipo_recomendado(atributo):
    if atributo == "order_item_id":
        return "INTEGER"
    if atributo.endswith("_id") or atributo in {"review_id", "order_id"}:
        return "VARCHAR"
    if "zip_code_prefix" in atributo:
        return "CHAR(5)"
    if atributo.endswith(("_timestamp", "_date", "_at")):
        return "TIMESTAMP"
    if atributo in {"price", "freight_value", "payment_value"}:
        return "DECIMAL(12,2)"
    if atributo in {
        "payment_installments",
        "payment_sequential",
        "review_score",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    }:
        return "INTEGER"
    return "VARCHAR"


inventario_atributos = pd.DataFrame(
    [
        {
            "Entidade": nome,
            "Atributo original": atributo,
            "Tipo inferido": str(df[atributo].dtype),
            "Tipo recomendado": tipo_recomendado(atributo),
            "Registros": len(df),
            "Valores ausentes": int(df[atributo].isna().sum()),
            "Ausentes (%)": df[atributo].isna().mean() * 100,
            "Valores únicos": int(df[atributo].nunique(dropna=True)),
            "Aderência": (
                "Previsto nos requisitos"
                if atributo in atributos_esperados[nome]
                else (
                    "Fora do escopo"
                    if (nome, atributo) in atributos_fora_do_escopo
                    else "Adicional relevante"
                )
            ),
            "Decisão": (
                "Descartar"
                if (nome, atributo) in atributos_fora_do_escopo
                else "Manter"
            ),
            "Justificativa": (
                "Texto livre não estruturado, excluído do escopo do projeto."
                if (nome, atributo) in atributos_fora_do_escopo
                else justificativas_adicionais.get(
                    (nome, atributo),
                    "Atributo aderente ao dicionário de dados e ao domínio.",
                )
            ),
        }
        for nome, df in entidades.items()
        for atributo in df.columns
    ]
)

display(
    inventario_atributos.style.format(
        {
            "Registros": "{:,.0f}",
            "Valores ausentes": "{:,.0f}",
            "Ausentes (%)": "{:.2f}%",
            "Valores únicos": "{:,.0f}",
        }
    ).hide(axis="index")
)

Entidade,Atributo original,Tipo inferido,Tipo recomendado,Registros,Valores ausentes,Ausentes (%),Valores únicos,Aderência,Decisão,Justificativa
Clientes,customer_id,str,VARCHAR,"99,441",0,0.00%,"99,441",Previsto nos requisitos,Manter,Atributo aderente ao dicionário de dados e ao domínio.
Clientes,customer_unique_id,str,VARCHAR,"99,441",0,0.00%,"96,096",Previsto nos requisitos,Manter,Atributo aderente ao dicionário de dados e ao domínio.
Clientes,customer_zip_code_prefix,int64,CHAR(5),"99,441",0,0.00%,"14,994",Previsto nos requisitos,Manter,Atributo aderente ao dicionário de dados e ao domínio.
Clientes,customer_city,str,VARCHAR,"99,441",0,0.00%,"4,119",Previsto nos requisitos,Manter,Atributo aderente ao dicionário de dados e ao domínio.
Clientes,customer_state,str,VARCHAR,"99,441",0,0.00%,27,Previsto nos requisitos,Manter,Atributo aderente ao dicionário de dados e ao domínio.
Pedidos,order_id,str,VARCHAR,"99,441",0,0.00%,"99,441",Previsto nos requisitos,Manter,Atributo aderente ao dicionário de dados e ao domínio.
Pedidos,customer_id,str,VARCHAR,"99,441",0,0.00%,"99,441",Previsto nos requisitos,Manter,Atributo aderente ao dicionário de dados e ao domínio.
Pedidos,order_status,str,VARCHAR,"99,441",0,0.00%,8,Previsto nos requisitos,Manter,Atributo aderente ao dicionário de dados e ao domínio.
Pedidos,order_purchase_timestamp,str,TIMESTAMP,"99,441",0,0.00%,"98,875",Previsto nos requisitos,Manter,Atributo aderente ao dicionário de dados e ao domínio.
Pedidos,order_approved_at,str,TIMESTAMP,"99,441",160,0.16%,"90,733",Previsto nos requisitos,Manter,Atributo aderente ao dicionário de dados e ao domínio.


## 4. Cobertura dos requisitos

A cobertura verifica se todos os atributos esperados estão disponíveis. Atributos
adicionais não representam falhas: eles são mantidos quando contribuem para a
modelagem ou descartados quando contrariam o escopo.

In [4]:
cobertura_requisitos = pd.DataFrame(
    [
        {
            "Entidade": nome,
            "Esperados": len(esperados),
            "Encontrados": len(esperados & set(entidades[nome].columns)),
            "Ausentes": ", ".join(sorted(esperados - set(entidades[nome].columns)))
            or "Nenhum",
            "Adicionais": len(set(entidades[nome].columns) - esperados),
            "Cobertura (%)": (
                len(esperados & set(entidades[nome].columns)) / len(esperados) * 100
            ),
        }
        for nome, esperados in atributos_esperados.items()
    ]
)

display(
    cobertura_requisitos.style.format({"Cobertura (%)": "{:.0f}%"}).hide(
        axis="index"
    )
)

Entidade,Esperados,Encontrados,Ausentes,Adicionais,Cobertura (%)
Clientes,5,5,Nenhum,0,100%
Pedidos,8,8,Nenhum,0,100%
Itens do Pedido,6,6,Nenhum,1,100%
Produtos,6,6,Nenhum,3,100%
Vendedores,4,4,Nenhum,0,100%
Pagamentos,4,4,Nenhum,1,100%
Avaliações,4,4,Nenhum,3,100%


## 5. Validação dos identificadores e chaves candidatas

A unicidade é avaliada sobre os dados completos. Para entidades com múltiplos
registros por pedido, a chave precisa ser composta.

In [5]:
chaves_candidatas = [
    ("Clientes", ["customer_id"], "Chave primária"),
    ("Clientes", ["customer_unique_id"], "Identificador de negócio"),
    ("Pedidos", ["order_id"], "Chave primária"),
    ("Itens do Pedido", ["order_id", "order_item_id"], "Chave primária composta"),
    ("Produtos", ["product_id"], "Chave primária"),
    ("Vendedores", ["seller_id"], "Chave primária"),
    ("Pagamentos", ["order_id", "payment_sequential"], "Chave primária composta"),
    ("Avaliações", ["review_id"], "Candidato simples"),
    ("Avaliações", ["order_id"], "Candidato simples"),
    ("Avaliações", ["review_id", "order_id"], "Chave primária composta"),
]

validacao_chaves = pd.DataFrame(
    [
        {
            "Entidade": nome,
            "Atributo(s)": " + ".join(atributos),
            "Papel avaliado": papel,
            "Nulos": int(df[atributos].isna().any(axis=1).sum()),
            "Registros duplicados": int(
                df.duplicated(subset=atributos, keep=False).sum()
            ),
            "É única": not df.duplicated(subset=atributos).any(),
        }
        for nome, atributos, papel in chaves_candidatas
        for df in [entidades[nome]]
    ]
)

validacao_chaves["Conclusão"] = validacao_chaves.apply(
    lambda linha: (
        "Chave válida"
        if linha["É única"] and linha["Nulos"] == 0
        else (
            "Identifica o consumidor, não a linha"
            if linha["Atributo(s)"] == "customer_unique_id"
            else "Requer composição com outro atributo"
        )
    ),
    axis=1,
)

display(
    validacao_chaves.drop(columns="É única")
    .style.format(
        {
            "Nulos": "{:,.0f}",
            "Registros duplicados": "{:,.0f}",
        }
    )
    .hide(axis="index")
)

Entidade,Atributo(s),Papel avaliado,Nulos,Registros duplicados,Conclusão
Clientes,customer_id,Chave primária,0,0,Chave válida
Clientes,customer_unique_id,Identificador de negócio,0,"6,342","Identifica o consumidor, não a linha"
Pedidos,order_id,Chave primária,0,0,Chave válida
Itens do Pedido,order_id + order_item_id,Chave primária composta,0,0,Chave válida
Produtos,product_id,Chave primária,0,0,Chave válida
Vendedores,seller_id,Chave primária,0,0,Chave válida
Pagamentos,order_id + payment_sequential,Chave primária composta,0,0,Chave válida
Avaliações,review_id,Candidato simples,0,"1,603",Requer composição com outro atributo
Avaliações,order_id,Candidato simples,0,"1,098",Requer composição com outro atributo
Avaliações,review_id + order_id,Chave primária composta,0,0,Chave válida


## 6. Pontos de atenção de qualidade

Valores ausentes não invalidam automaticamente um atributo. Datas posteriores à
compra podem estar vazias devido ao estado do pedido, enquanto atributos físicos
de produtos representam lacunas da fonte. Esses casos deverão orientar as
restrições de nulidade no modelo lógico.

In [6]:
pontos_atencao = (
    inventario_atributos.loc[
        (inventario_atributos["Valores ausentes"] > 0)
        & (inventario_atributos["Decisão"] == "Manter"),
        [
            "Entidade",
            "Atributo original",
            "Valores ausentes",
            "Ausentes (%)",
            "Tipo recomendado",
        ],
    ]
    .sort_values(["Ausentes (%)", "Entidade"], ascending=[False, True])
    .reset_index(drop=True)
)

display(
    pontos_atencao.style.format(
        {
            "Valores ausentes": "{:,.0f}",
            "Ausentes (%)": "{:.2f}%",
        }
    ).hide(axis="index")
)

Entidade,Atributo original,Valores ausentes,Ausentes (%),Tipo recomendado
Pedidos,order_delivered_customer_date,"2,965",2.98%,TIMESTAMP
Produtos,product_category_name,610,1.85%,VARCHAR
Produtos,product_name_lenght,610,1.85%,INTEGER
Produtos,product_description_lenght,610,1.85%,INTEGER
Produtos,product_photos_qty,610,1.85%,INTEGER
Pedidos,order_delivered_carrier_date,"1,783",1.79%,TIMESTAMP
Pedidos,order_approved_at,160,0.16%,TIMESTAMP
Produtos,product_weight_g,2,0.01%,INTEGER
Produtos,product_length_cm,2,0.01%,INTEGER
Produtos,product_height_cm,2,0.01%,INTEGER


## 7. Resultado final

A validação confirma que:

- todas as sete entidades possuem 100% dos atributos previstos nos requisitos;
- 45 atributos foram encontrados nas fontes analisadas;
- 43 atributos serão mantidos e 2 campos de comentários serão descartados por
  estarem fora do escopo;
- Clientes, Pedidos, Produtos e Vendedores possuem chaves simples válidas;
- Itens do Pedido e Pagamentos exigem chaves compostas;
- Avaliações exige a composição de `review_id` com `order_id`, pois nenhum dos
  dois campos é único isoladamente;
- `customer_unique_id` representa o consumidor ao longo de diferentes compras,
  mas não substitui `customer_id` como identificador da linha;
- prefixos de CEP devem ser tratados como `CHAR(5)`, evitando a perda de zeros à
  esquerda;
- campos temporais devem ser convertidos de texto para `TIMESTAMP` na carga;
- atributos mantidos com valores ausentes deverão aceitar nulidade de acordo com
  as regras do processo de negócio.

Com essas decisões, os atributos estão validados para a definição dos
relacionamentos, cardinalidades e restrições do modelo conceitual.